In [1]:

from pathlib import Path

# إنشاء مجلد المشروع الرئيسي
ROOT = Path("/kaggle/working/pdpl_compliance_copilot")

# إنشاء المجلدات الفرعية
folders = [
    "src",
    "configs",
    "data/raw",
    "data/processed",
    "eval",
    "notebooks",
]

for folder in folders:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print("تم إنشاء مجلدات المشروع بنجاح")
print(ROOT)


تم إنشاء مجلدات المشروع بنجاح
/kaggle/working/pdpl_compliance_copilot


In [2]:

for path in sorted(ROOT.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(ROOT))


📁 configs
📁 data
📁 data/processed
📁 data/raw
📁 eval
📁 notebooks
📁 src


In [3]:

# ملفات الكود الأساسية
source_files = [
    "src/__init__.py",
    "src/ingestion.py",
    "src/retrieval.py",
    "src/generation.py",
    "src/classifier.py",
    "src/gap_analyzer.py",
    "configs/settings.yaml",
    "eval/gold_qa_set.json",
    "eval/local_policies_annotated.json",
    "README.md",
    "requirements.txt",
]

for name in source_files:
    path = ROOT / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.touch()

# دفاتر المراحل التالية
notebooks = [
    "02_hybrid_retrieval_and_rerank.ipynb",
    "03_clause_classifier_training.ipynb",
    "04_rag_and_gap_analysis.ipynb",
    "05_end_to_end_evaluation.ipynb",
]

import nbformat

for name in notebooks:
    path = ROOT / "notebooks" / name
    if not path.exists():
        nb = nbformat.v4.new_notebook()
        nb.cells.append(
            nbformat.v4.new_markdown_cell(
                f"# {name.replace('.ipynb', '').replace('_', ' ')}"
            )
        )
        nbformat.write(nb, path)

print("تم إنشاء الملفات الأساسية والدفاتر التالية.")


تم إنشاء الملفات الأساسية والدفاتر التالية.



# المرحلة الأولى: تجهيز البيانات ومقارنة استراتيجيات التقسيم

## الهدف

تهدف هذه المرحلة إلى تجهيز مجموعات البيانات العربية والإنجليزية الخاصة بنظام حماية البيانات الشخصية (PDPL)، والتحقق من بنيتها وجودتها، ومقارنة طريقتَي تقسيم النصوص: التقسيم الثابت (`Fixed-size Chunking`) والتقسيم حسب المادة أو القسم (`Article/Section Chunking`).

## نطاق العمل

1. تحميل مجموعات البيانات والتحقق من الحقول الأساسية.
2. تطبيع النصوص العربية لأغراض البحث مع الاحتفاظ بالنص الأصلي.
3. تجهيز المقاطع والبيانات الوصفية.
4. مقارنة عدد المقاطع وأطوالها وتوفر المعرّفات وأرقام المواد.
5. فحص المقاطع القصيرة والتحقق من سلامة المعرّفات.
6. حفظ المخرجات والتحقق من إمكانية قراءتها مجددًا.

## مجموعات البيانات

تتضمن البيانات أربع مجموعات للمقاطع العربية والإنجليزية، بالإضافة إلى مجموعة أسئلة وإجابات مرجعية (`Gold QA`) لاستخدامها في مرحلة التقييم اللاحقة.

## حدود المرحلة

تركز هذه المرحلة على تجهيز البيانات وفحصها ومقارنة استراتيجيات التقسيم. لا تتضمن بناء نظام الاسترجاع أو توليد الإجابات أو تقييم أداء النظام النهائي.


In [4]:

from pathlib import Path

BASE = Path("/kaggle/input/datasets/khadijah2000")

FILES = {
    "ar_fixed": BASE / "pdpl-chunking-comparison-datasets/fixed_size_chunks.json",
    "ar_article": BASE / "pdpl-chunking-comparison-datasets/article_section_chunks.json",
    "en_fixed": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json",
    "en_article": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json",
    "gold_qa": BASE / "pdpl-bilingual-evaluation-dataset/gold_qa (2).json",
}

for name, path in FILES.items():
    print("✅" if path.is_file() else "❌", name, "—", path)


✅ ar_fixed — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets/fixed_size_chunks.json
✅ ar_article — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets/article_section_chunks.json
✅ en_fixed — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json
✅ en_article — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json
✅ gold_qa — /kaggle/input/datasets/khadijah2000/pdpl-bilingual-evaluation-dataset/gold_qa (2).json


## 1. تحميل البيانات والتحقق من بنيتها

### الهدف
تحميل مجموعات البيانات العربية والإنجليزية الخاصة بمشروع PDPL، والتحقق من عدد السجلات والوثائق والحقول المتاحة قبل بدء تجهيز النصوص.

### مجموعات البيانات
- `ar_fixed`: المقاطع العربية ثابتة الحجم.
- `ar_article`: المقاطع العربية حسب المادة أو القسم.
- `en_fixed`: المقاطع الإنجليزية ثابتة الحجم.
- `en_article`: المقاطع الإنجليزية حسب المادة أو القسم.
- `gold_qa`: أسئلة وإجابات مرجعية لاستخدامها في مرحلة التقييم اللاحقة.


In [5]:

# المرحلة 1: تحميل البيانات وفحص بنيتها

import json
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/input/datasets/khadijah2000")

FILES = {
    "ar_fixed": BASE / "pdpl-chunking-comparison-datasets/fixed_size_chunks.json",
    "ar_article": BASE / "pdpl-chunking-comparison-datasets/article_section_chunks.json",
    "en_fixed": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json",
    "en_article": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json",
    "gold_qa": BASE / "pdpl-bilingual-evaluation-dataset/gold_qa (2).json",
}

raw_data = {}

for name, path in FILES.items():
    with path.open("r", encoding="utf-8") as f:
        raw_data[name] = json.load(f)
    print(f"تم تحميل {name}")

def get_records(data):
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("records", "chunks", "data", "items"):
            if isinstance(data.get(key), list):
                return data[key]

        return [data]

    return []

# نبقي المجموعات منفصلة ولا ندمجها الآن
retrieval_data = {
    name: get_records(raw_data[name])
    for name in ("ar_fixed", "ar_article", "en_fixed", "en_article")
}

summary = []

for name, records in retrieval_data.items():
    first = records[0] if records else {}

    text_fields = [
        field
        for field in ("search_text", "clean_text", "text")
        if any(str(r.get(field, "")).strip() for r in records)
    ]

    doc_ids = {
        r.get("document_id") or r.get("parent_id")
        for r in records
        if r.get("document_id") or r.get("parent_id")
    }

    summary.append({
        "dataset": name,
        "chunks": len(records),
        "documents_with_ids": len(doc_ids),
        "text_fields_found": ", ".join(text_fields),
        "article_number_field": "article_number" in first,
        "covered_article_numbers_field": "covered_article_numbers" in first,
        "sample_fields": ", ".join(first.keys()) if first else "لا توجد سجلات",
    })

display(pd.DataFrame(summary))

gold_records = get_records(raw_data["gold_qa"])
print("عدد سجلات gold_qa:", len(gold_records))

if gold_records:
    print("حقول أول سؤال:", list(gold_records[0].keys()))
    print("عينة من السؤال الأول:")
    print(json.dumps(gold_records[0], ensure_ascii=False, indent=2)[:2000])


تم تحميل ar_fixed
تم تحميل ar_article
تم تحميل en_fixed
تم تحميل en_article
تم تحميل gold_qa


,dataset,chunks,documents_with_ids,text_fields_found,article_number_field,covered_article_numbers_field,sample_fields
0,ar_fixed,241,10,"search_text, clean_text, text",True,False,"id, document_id, language, article_number, art..."
1,ar_article,196,10,"search_text, clean_text, text",True,False,"id, document_id, language, article_number, art..."
2,en_fixed,181,10,"search_text, clean_text",True,True,"chunk_id, document_id, language, chunking, chu..."
3,en_article,252,9,"search_text, clean_text",True,False,"chunk_id, parent_id, document_id, language, un..."


عدد سجلات gold_qa: 90
حقول أول سؤال: ['question_id', 'language', 'question', 'reference_answer', 'gold_evidence_extract', 'source_workbook', 'source_document', 'source_id_original', 'gold_reference_original', 'gold_document_id', 'gold_article_number', 'gold_references', 'answerable', 'in_corpus', 'evaluation_status', 'eligible_for_article_retrieval_eval', 'reference_type', 'question_type', 'pdf_page', 'review_notes', 'question_v2', 'revision_action', 'revision_reason', 'evidence_matching_status', 'reference_review_status', 'review_priority', 'reviewer_notes', 'approved_for_final_eval', 'best_chunk_id', 'evidence_review_status', 'gold_reference_in_chunk_corpus', 'manual_evidence_approval']
عينة من السؤال الأول:
{
  "question_id": "AR001",
  "language": "ar",
  "question": "ما المقصود بالبيانات الشخصية في النظام؟",
  "reference_answer": "كل بيان يمكن أن يؤدي إلى معرفة الفرد أو يجعل التعرف عليه ممكنًا مباشرة أو بصورة غير مباشرة، بصرف النظر عن مصدره أو شكله.",
  "gold_evidence_extract": "ك


### استنتاج الخطوة الأولى

تم تحميل ملفات البيانات الخمسة بنجاح. تحتوي مجموعات المقاطع الأربع على إجمالي 870 مقطعًا، موزعة على 241 مقطعًا عربيًا ثابت الحجم، و196 مقطعًا عربيًا حسب المادة، و181 مقطعًا إنجليزيًا ثابت الحجم، و252 مقطعًا إنجليزيًا حسب المادة.

كما تم تحميل 90 سؤالًا في مجموعة `Gold QA`. ستُحفظ هذه المجموعة لاستخدامها في التقييم اللاحق، دون تحليل مراجعها في هذه المرحلة.

أظهرت معاينة الحقول اختلافًا بين بنية المجموعات، لذلك سنحافظ على كل مجموعة منفصلة أثناء تجهيز البيانات ومقارنة استراتيجيات التقسيم.


In [11]:

# المرحلة 2.2: إعادة فحص المراجع بعد توحيد تنسيق رقم المادة

def normalize_article(value):
    if pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value).strip()

chunks_check = chunks_df.copy()

chunks_check["document_id_norm"] = (
    chunks_check["document_id"].fillna("").astype(str).str.strip().str.lower()
)
chunks_check["article_norm"] = chunks_check["article_number"].apply(
    normalize_article
)
chunks_check["language_norm"] = (
    chunks_check["language"].fillna("").astype(str).str.strip().str.lower()
)

gold_check = gold_df.copy()

gold_check["document_norm"] = (
    gold_check["gold_document_id"].fillna("").astype(str).str.strip().str.lower()
)
gold_check["article_norm"] = gold_check["gold_article_number"].apply(
    normalize_article
)
gold_check["language_norm"] = (
    gold_check["language"].fillna("").astype(str).str.strip().str.lower()
)

def check_reference(row):
    candidates = chunks_check[
        (chunks_check["document_id_norm"] == row["document_norm"])
        & (chunks_check["article_norm"] == row["article_norm"])
        & (chunks_check["language_norm"] == row["language_norm"])
    ]
    return len(candidates) > 0

gold_check["article_reference_found_normalized"] = gold_check.apply(
    check_reference, axis=1
)

print("نتيجة المطابقة بعد توحيد التنسيق:")
print(
    gold_check.groupby("language_norm")[
        "article_reference_found_normalized"
    ].agg(["count", "sum"])
)

print("\nتوزيع الوثائق في مجموعات المقاطع:")
display(
    chunks_check.groupby(
        ["dataset", "document_id_norm", "language_norm"]
    ).size().reset_index(name="chunks")
)

print("\nعينة من المراجع التي لم نجد لها تطابقًا:")
display(
    gold_check.loc[
        ~gold_check["article_reference_found_normalized"],
        [
            "question_id", "language_norm", "document_norm",
            "article_norm", "best_chunk_id"
        ]
    ].head(15)
)


نتيجة المطابقة بعد توحيد التنسيق:
               count  sum
language_norm            
ar                60   45
en                30   24

توزيع الوثائق في مجموعات المقاطع:


,dataset,document_id_norm,language_norm,chunks
0,ar_article,accreditation_certificates,ar,15
1,ar_article,accreditation_licensing,ar,20
2,ar_article,appointing_dpo,ar,11
3,ar_article,data_monetization_policy,ar,15
4,ar_article,data_sharing_policy,ar,19
5,ar_article,hrsd_780009_en,ar,10
6,ar_article,implementing_regulation,ar,42
7,ar_article,pdpl,ar,44
8,ar_article,secondary_data_use,ar,9
9,ar_article,transfer_regulation,ar,11



عينة من المراجع التي لم نجد لها تطابقًا:


,question_id,language_norm,document_norm,article_norm,best_chunk_id
45,AR046,ar,breach_response_guide,,None
46,AR047,ar,breach_response_guide,,None
47,AR048,ar,breach_response_guide,,None
48,AR049,ar,breach_response_guide,,None
49,AR050,ar,breach_response_guide,,None
50,AR051,ar,,,None
51,AR052,ar,,,None
52,AR053,ar,,,None
53,AR054,ar,,,None
54,AR055,ar,,,None



### المرحلة 2.2: التحقق من المراجع في مجموعة Gold QA

#### الهدف
التحقق من وجود المراجع القانونية المرتبطة بأسئلة التقييم داخل مجموعة المقاطع، بعد توحيد تنسيق أرقام المواد ومعرّفات الوثائق واللغة.

#### النتائج الأولية

| اللغة | إجمالي الأسئلة | المراجع المطابقة | غير المطابقة |
|---|---:|---:|---:|
| العربية | 60 | 45 | 15 |
| الإنجليزية | 30 | 24 | 6 |
| **الإجمالي** | **90** | **69** | **21** |

بلغت نسبة العثور على تطابق مع مرجع قانوني ضمن شروط المطابقة الحالية 76.7% (69 من أصل 90 سؤالًا).

#### تفسير النتائج

عدم العثور على مرجع مطابق لا يعني تلقائيًا وجود خطأ في مجموعة البيانات. تتضمن مجموعة التقييم أسئلة صُممت لاختبار حدود النظام، بما فيها الأسئلة التي لا يغطيها المحتوى المتاح، وذلك لقياس قدرة النظام على الامتناع عن تقديم إجابات غير مدعومة بالأدلة.

لذلك، يجب التمييز بين الأسئلة القابلة للإجابة التي يُتوقع وجود مراجع لها، والأسئلة الاختبارية غير القابلة للإجابة أو التي تقع خارج نطاق قاعدة البيانات. تُحفظ الفئة الثانية ضمن مجموعة التقييم لاستخدامها في اختبار الامتناع عن الإجابة (Abstention).

#### ملاحظة منهجية

تمثل هذه النتائج فحصًا أوليًا لمطابقة المراجع، وليست مقياسًا نهائيًا لأداء الاسترجاع. قبل اعتماد النتائج النهائية، تُراجع حالة كل سؤال وحقول `answerable` و`in_corpus` و`eligible_for_article_retrieval_eval` و`reference_type` للتأكد من تصنيفه ضمن التقييم المناسب.



## المرحلة 3: منهجية بناء النظام وتقييمه

### الهدف
بناء PDPL Compliance Copilot ثنائي اللغة (العربية والإنجليزية)، يعتمد على استرجاع الأدلة القانونية قبل صياغة الإجابة، ويُظهر مصدر الاستدلال أو يمتنع عن الإجابة عند عدم كفاية الأدلة.

### مجموعات البيانات
نعتمد على أربع مجموعات موجودة مسبقًا:
- Arabic Fixed-size Chunks
- Arabic Article-level Chunks
- English Fixed-size Chunks
- English Article-level Chunks

تُستخدم المجموعات لمقارنة أثر استراتيجية تقسيم النصوص على جودة الاسترجاع، مع إبقاء النص القانوني الأصلي ومعلومات المصدر محفوظة.

### تصميم مجموعة Gold QA
تتضمن مجموعة التقييم أسئلة مدعومة بالمراجع وأسئلة لا يتوفر لها مرجع أو دليل ضمن البيانات المستخدمة. أُدرجت بعض الأسئلة العربية والإنجليزية لاختبار قدرة النظام على الامتناع عن الإجابة عندما لا تكفي الأدلة المتاحة.

لذلك، لا يُعد غياب المرجع خطأً تلقائيًا. ويجب التمييز بين:
1. سؤال مدعوم بدليل متاح.
2. سؤال لا يتوفر دليله ضمن مجموعة البيانات.
3. سؤال يحتاج إلى مراجعة مرجعه أو حالة اعتماده.

غياب الدليل عن مجموعة البيانات لا يعني بالضرورة عدم وجود إجابة في المصادر القانونية خارجها.

### آلية التقييم
سيُقيّم النظام على محورين منفصلين:

- **Retrieval Evaluation:** قدرة النظام على استرجاع المقاطع القانونية ذات الصلة.
- **Abstention Evaluation:** قدرة النظام على الامتناع عن تقديم إجابة مؤكدة عندما لا تتوفر أدلة كافية.

لن تُعامل الأسئلة المصممة لاختبار الامتناع باعتبارها أسئلة استرجاع فاشلة تلقائيًا، ولن تُختلق لها مراجع أو إجابات مرجعية.

### سلوك النظام المطلوب
- الاستناد إلى النصوص القانونية المسترجعة في صياغة الإجابة.
- إظهار الوثيقة ورقم المادة والمقطع الداعم متى توفرت البيانات.
- التصريح بعدم كفاية الأدلة عند تعذر دعم الإجابة.
- عدم اختلاق مادة قانونية أو مصدر غير موجود.
- الفصل بين نتائج الاسترجاع ونتائج الامتناع.

### الحالة الحالية
تم تحميل مجموعات المقاطع الأربع و90 سجلًا من Gold QA، وأُجري فحص أولي لبنية البيانات وتوافق المراجع.



## 2. تطبيع النصوص وتجهيزها للبحث

### الهدف
تجهيز النصوص القانونية للاستخدام في عمليات البحث والاسترجاع، من خلال توحيد بعض الاختلافات الكتابية في النصوص العربية، مع الحفاظ على النص الأصلي دون تعديل.

### خطوات المعالجة
1. تطبيق Unicode Normalization باستخدام `NFKC`.
2. إزالة التشكيل والتطويل من النص العربي.
3. توحيد أشكال الألف (`أ، إ، آ، ٱ`) إلى `ا`، وتوحيد `ى` إلى `ي` لأغراض البحث.
4. توحيد المسافات الزائدة.
5. إنشاء نسخة نصية مطبّعة في حقل مستقل، مع الاحتفاظ بالنص الأصلي في نسخة العمل.

### ملاحظة منهجية
يُطبّق التطبيع على نسخة مخصصة للبحث فقط، ولا تُستبدل به الصياغة القانونية الأصلية. ويتيح ذلك الاستفادة من النص المطبّع في البحث، مع الرجوع إلى النص الأصلي عند عرض المواد القانونية والاستشهاد بها.

In [18]:

# المرحلة 1.2: تطبيع النص العربي للبحث مع حفظ النص الأصلي

import re
import unicodedata

def normalize_arabic_for_search(text, normalize_ta_marbuta=False):
    """
    تطبيع النص لأغراض البحث فقط، دون تعديل النص القانوني الأصلي.
    يمكن تفعيل توحيد التاء المربوطة مع الهاء عند اختبار أثره.
    """
    text = unicodedata.normalize("NFKC", str(text or ""))

    # إزالة التشكيل والتطويل
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.replace("ـ", "")

    # توحيد أشكال الألف
    text = re.sub(r"[أإآٱ]", "ا", text)

    # توحيد الألف المقصورة مع الياء
    text = text.replace("ى", "ي")

    # خيار اختياري لتوحيد التاء المربوطة مع الهاء
    if normalize_ta_marbuta:
        text = text.replace("ة", "ه")

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text


# إنشاء نسخ للعمل دون تعديل مجموعات البيانات الأصلية
normalized_data = {}

for dataset_name, records in retrieval_data.items():
    normalized_records = []

    for record in records:
        item = record.copy()

        # اختيار النص المتاح مع الاحتفاظ بنسخته الأصلية
        original_text = (
            record.get("search_text")
            or record.get("clean_text")
            or record.get("text")
            or ""
        )

        item["original_text_preserved"] = original_text

        # تطبيع النص العربي فقط، مع إبقاء النص الإنجليزي كما هو
        if str(record.get("language", "")).lower().startswith("ar"):
            item["normalized_search_text"] = normalize_arabic_for_search(
                original_text,
                normalize_ta_marbuta=False
            )
        else:
            item["normalized_search_text"] = str(original_text).strip()

        normalized_records.append(item)

    normalized_data[dataset_name] = normalized_records


# التحقق من عدد المقاطع والحفاظ على النص الأصلي
print("اكتمل التطبيع دون تعديل مجموعات البيانات الأصلية.")

for name, records in normalized_data.items():
    original_records = retrieval_data[name]

    assert len(records) == len(original_records), (
        f"عدد المقاطع غير متطابق في المجموعة: {name}"
    )

    for original, normalized in zip(original_records, records):
        expected_text = (
            original.get("search_text")
            or original.get("clean_text")
            or original.get("text")
            or ""
        )

        assert normalized["original_text_preserved"] == expected_text, (
            f"تغير النص الأصلي في المجموعة: {name}"
        )

    print(f"✅ {name}: {len(records)} مقطع، والنص الأصلي محفوظ")


اكتمل التطبيع دون تعديل مجموعات البيانات الأصلية.
✅ ar_fixed: 241 مقطع، والنص الأصلي محفوظ
✅ ar_article: 196 مقطع، والنص الأصلي محفوظ
✅ en_fixed: 181 مقطع، والنص الأصلي محفوظ
✅ en_article: 252 مقطع، والنص الأصلي محفوظ


In [13]:

# التحقق من التطبيع والحفاظ على النص القانوني الأصلي

sample = normalized_data["ar_article"][0]

original = sample["original_text_preserved"]
normalized = sample["normalized_search_text"]

print("النص الأصلي:")
print(original[:500])

print("\nالنص المطبّع للبحث:")
print(normalized[:500])

print("\nنتائج التحقق:")
print("النص الأصلي محفوظ دون تغيير:", 
      original == (
          sample.get("search_text")
          or sample.get("clean_text")
          or sample.get("text")
          or ""
      ))
print("حقل النص المطبّع موجود:", "normalized_search_text" in sample)
print("عدد حروف النص الأصلي:", len(original))
print("عدد حروف النص المطبّع:", len(normalized))


النص الأصلي:
لغرض تطبيق هذا النظام، يُقصد بالألفاظ والعبارات الآتية -أينما وردت في هذا النظام- المعاني المبينة أمام كل منها، ما لم يقتضِ السياق غير ذلك: 1- النظام : نظام حماية البيانات الشخصية. 2- اللوائح : اللوائح التنفيذية للنظام. 3- الجهة المختصة : الجهة التي يصدر بتحديدها قرار من مجلس الوزراء. 4- البيانات الشخصية : كل بيان -مهما كان مصدره أو شكله- من شأنه أن يؤدي إلى معرفة الفرد على وجه التحديد، أو يجعل التعرف عليه ممكنًا بصفة مباشرة أو غير مباشرة، ومن ذلك: الاسم، ورقم الهوية الشخصية، والعناوين، وأرقام ا

النص المطبّع للبحث:
لغرض تطبيق هذا النظام، يقصد بالالفاظ والعبارات الاتية -اينما وردت في هذا النظام- المعاني المبينة امام كل منها، ما لم يقتض السياق غير ذلك: 1- النظام : نظام حماية البيانات الشخصية. 2- اللوائح : اللوائح التنفيذية للنظام. 3- الجهة المختصة : الجهة التي يصدر بتحديدها قرار من مجلس الوزراء. 4- البيانات الشخصية : كل بيان -مهما كان مصدره او شكله- من شانه ان يؤدي الي معرفة الفرد علي وجه التحديد، او يجعل التعرف عليه ممكنا بصفة مباشرة او غير مباشرة، ومن ذلك: الاسم، ورقم اله

### استنتاج المرحلة الثانية

تم تطبيق التطبيع على نسخة مخصصة للبحث من النصوص العربية، مع الحفاظ على النص القانوني الأصلي دون تغيير.

أظهرت نتائج التحقق ما يلي:
- **سلامة النص الأصلي:** بقي النص الأصلي مطابقًا للمصدر المستخدم في المعالجة.
- **توفر النص المطبّع:** تم إنشاء الحقل `normalized_search_text` بنجاح.
- **اختلاف طول النص:** انخفض عدد الأحرف في العينة من 1645 إلى 1628 حرفًا نتيجة إزالة التشكيل والتطويل وتوحيد بعض الحروف والمسافات.

تؤكد هذه النتائج نجاح التطبيع في العينة التي تم فحصها. وسيُستخدم النص المطبّع لأغراض البحث، مع الاحتفاظ بالنص الأصلي للرجوع إلى الصياغة القانونية الدقيقة.

**الخطوة التالية:** تجهيز البيانات الوصفية (Metadata) للمقاطع القانونية لتسهيل تنظيمها ومقارنتها.

## 3. تجهيز المقاطع والبيانات الوصفية (Metadata)

### الهدف
تجهيز المقاطع الموجودة مسبقًا في مجموعتي التقسيم حسب المادة أو القسم (Article-level) العربية والإنجليزية، وإضافة بيانات وصفية منظمة تساعد في تحديد مصدر كل مقطع وربطه بوثيقته ومادته القانونية.

### البيانات التي ستُجهّز
تتضمن البيانات الوصفية:
- `chunk_id`: معرّف المقطع.
- `document_id` و`parent_id`: معرّفات الوثيقة والوثيقة الأصلية عند توفرها.
- `article_number`: رقم المادة القانونية.
- `language`: لغة المقطع.
- `source_url`: رابط المصدر.
- `source_record_id`: معرّف السجل الأصلي عند توفره.

### نطاق المعالجة
سيتم الاحتفاظ بالنص الأصلي والنص المطبّع للبحث، وإضافة البيانات الوصفية داخل حقل مستقل باسم `metadata`، دون إعادة تقسيم النصوص أو دمج مجموعات البيانات الأربع.

تُجهّز في هذه الخطوة مجموعتا Article-level فقط، بينما ستبقى مجموعتا Fixed-size منفصلتين للمقارنة في الخطوة التالية.

In [14]:

# المرحلة 3: تجهيز المقاطع والبيانات الوصفية

article_data = {}

for dataset_name in ("ar_article", "en_article"):
    prepared_records = []

    for record in normalized_data[dataset_name]:
        item = record.copy()

        # توحيد اسم معرّف المقطع دون حذف المعرّفات الأصلية
        chunk_id = (
            item.get("chunk_id")
            or item.get("id")
            or item.get("source_record_id")
        )

        metadata = {
            "dataset": dataset_name,
            "chunk_id": chunk_id,
            "document_id": item.get("document_id"),
            "parent_id": item.get("parent_id"),
            "language": item.get("language"),
            "article_number": item.get("article_number"),
            "source_url": item.get("source_url"),
            "source_record_id": item.get("source_record_id"),
        }

        item["metadata"] = metadata
        prepared_records.append(item)

    article_data[dataset_name] = prepared_records

# عرض عينة للتحقق من النص والبيانات الوصفية
for name, records in article_data.items():
    print(f"\n--- {name} ---")
    print("عدد المقاطع:", len(records))

    if records:
        sample = records[0]
        print("معرّف المقطع:", sample["metadata"]["chunk_id"])
        print("الوثيقة:", sample["metadata"]["document_id"])
        print("رقم المادة:", sample["metadata"]["article_number"])
        print("اللغة:", sample["metadata"]["language"])
        print("رابط المصدر:", sample["metadata"]["source_url"])
        print("النص الأصلي محفوظ:", "original_text_preserved" in sample)
        print("النص المطبّع محفوظ:", "normalized_search_text" in sample)



--- ar_article ---
عدد المقاطع: 196
معرّف المقطع: pdpl_ar_001__article_section__001
الوثيقة: pdpl
رقم المادة: 1
اللغة: ar
رابط المصدر: https://dgp.sdaia.gov.sa/wps/portal/pdp/knowledgecenter/details/PDPL
النص الأصلي محفوظ: True
النص المطبّع محفوظ: True

--- en_article ---
عدد المقاطع: 252
معرّف المقطع: pdpl_001_chunk_01
الوثيقة: pdpl
رقم المادة: 1
اللغة: en
رابط المصدر: https://dgp.sdaia.gov.sa/wps/portal/pdp/knowledgecenter/details/PDPL/!ut/p/z1/04_Sj9CPykssy0xPLMnMz0vMAfIjo8ziPR1dzTwMgw2MDMOcTA3MjH39TE29jY0MQsz1w9EUhIZZAhUEGvl6OXoaGwQY60cRo98AB3A0IKTfi5ACoA-MinydfdP1owoSSzJ0M_PS8vUjAlwCfIB2R-HVbWGMoQDTe2AFeNxfkBsaUeWTFuyZrqgIALitKT0!/dz/d5/L0lDUmlTUSEhL3dHa0FKRnNBLzROV3FpQSEhL2Vu/
النص الأصلي محفوظ: True
النص المطبّع محفوظ: True


### استنتاج المرحلة الثالثة

تم تجهيز مجموعتي المقاطع المصنفة حسب المادة أو القسم، مع إنشاء حقل `metadata` لكل مقطع والاحتفاظ بالنص الأصلي والنص المطبّع للبحث.

بلغ إجمالي المقاطع المجهزة 448 مقطعًا، منها 196 مقطعًا عربيًا و252 مقطعًا إنجليزيًا. وأظهرت معاينة السجلات توفر بيانات وصفية أساسية، تشمل معرّف المقطع والوثيقة ورقم المادة واللغة ورابط المصدر.

تقتصر هذه الخطوة على تنظيم البيانات الموجودة، ولا تعني أن جميع حقول البيانات الوصفية مكتملة في كل المقاطع؛ وسيُفحص اكتمالها ضمن مقارنة مجموعات التقسيم.

**الخطوة التالية:** مقارنة مجموعات Article-level وFixed-size من حيث عدد المقاطع وأطوال النصوص وتوفر المعرّفات وأرقام المواد.


## 4. مقارنة استراتيجيات تقسيم النصوص

### الهدف
مقارنة مجموعات التقسيم الموجودة مسبقًا باللغتين العربية والإنجليزية، باستخدام استراتيجيتين: التقسيم حسب المادة أو القسم (Article-level)، والتقسيم ثابت الحجم (Fixed-size).

### مؤشرات المقارنة
- عدد المقاطع في كل مجموعة.
- متوسط طول النص وأقصر وأطول مقطع.
- نسبة المقاطع التي تحتوي على معرّفات.
- نسبة المقاطع التي تحتوي على معرّفات للوثائق.
- نسبة المقاطع التي تحتوي على أرقام المواد.
- عدد المقاطع الفارغة.

### منهجية المقارنة
تُحسب المؤشرات من النصوص الأصلية المحفوظة، مع إبقاء مجموعات البيانات الأربع منفصلة. وتُستخدم النتائج لوصف الفروق في بنية المقاطع واكتمال بياناتها الوصفية، دون اعتبارها دليلًا كافيًا على تفوق إحدى استراتيجيات التقسيم في أداء الاسترجاع.

In [15]:

# المرحلة 4: مقارنة استراتيجيات تقسيم النصوص

import pandas as pd

comparison_rows = []

for dataset_name, records in normalized_data.items():
    if not records:
        continue

    lengths = [
        len(str(
            record.get("original_text_preserved", "")
        ))
        for record in records
    ]

    comparison_rows.append({
        "dataset": dataset_name,
        "language": records[0].get("language"),
        "chunks": len(records),
        "avg_chars": round(sum(lengths) / len(lengths), 2),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "chunks_with_id_pct": round(
            100 * sum(
                bool(record.get("chunk_id") or record.get("id"))
                for record in records
            ) / len(records), 2
        ),
        "chunks_with_document_id_pct": round(
            100 * sum(
                bool(record.get("document_id"))
                for record in records
            ) / len(records), 2
        ),
        "chunks_with_article_number_pct": round(
            100 * sum(
                record.get("article_number") is not None
                and str(record.get("article_number")).strip() != ""
                for record in records
            ) / len(records), 2
        ),
        "empty_chunks": sum(
            not str(record.get("original_text_preserved", "")).strip()
            for record in records
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df[
    [
        "dataset",
        "language",
        "chunks",
        "avg_chars",
        "min_chars",
        "max_chars",
        "chunks_with_id_pct",
        "chunks_with_document_id_pct",
        "chunks_with_article_number_pct",
        "empty_chunks",
    ]
].sort_values("dataset").reset_index(drop=True)

display(comparison_df)

,dataset,language,chunks,avg_chars,min_chars,max_chars,chunks_with_id_pct,chunks_with_document_id_pct,chunks_with_article_number_pct,empty_chunks
0,ar_article,ar,196,746.63,6,2100,100.0,100.0,72.96,0
1,ar_fixed,ar,241,654.77,6,1319,100.0,100.0,71.37,0
2,en_article,en,252,733.42,9,1199,100.0,100.0,100.00,0
3,en_fixed,en,181,1166.51,160,1199,100.0,100.0,0.00,0


### نتائج مقارنة استراتيجيات التقسيم

أظهرت المقارنة وجود اختلافات بين مجموعات التقسيم العربية والإنجليزية من حيث عدد المقاطع ومتوسط طول النص وتوفر أرقام المواد.

- **المقاطع العربية:** بلغ متوسط طول `ar_article` نحو 746.63 حرفًا، مقابل 654.77 حرفًا في `ar_fixed`.
- **المقاطع الإنجليزية:** بلغ متوسط طول `en_article` نحو 733.42 حرفًا، مقابل 1166.51 حرفًا في `en_fixed`.
- **المعرّفات:** توفرت معرّفات المقاطع ومعرّفات الوثائق في جميع السجلات ضمن المجموعات الأربع وفق الفحص الحالي.
- **أرقام المواد:** توفرت في 72.96% من `ar_article` و71.37% من `ar_fixed` و100% من `en_article`، بينما لم تتوفر في أي سجل من `en_fixed` وفق الحقل `article_number` الذي فحصناه.
- **النصوص الفارغة:** لم تُسجّل مقاطع فارغة في أي مجموعة.

توضح النتائج أن بنية البيانات الوصفية تختلف بين مجموعات التقسيم، خصوصًا في توفر أرقام المواد. ولا تثبت هذه المؤشرات وحدها أن إحدى استراتيجيات التقسيم أفضل للاسترجاع؛ إذ يتطلب ذلك تقييمًا مستقلًا لاحقًا.

**الخطوة التالية:** فحص المقاطع القصيرة جدًا لمعرفة ما إذا كانت تحتوي على نصوص تحتاج إلى مراجعة قبل اعتماد البيانات.

In [19]:

# المرحلة 1.5: فحص المقاطع القصيرة جدًا

short_chunk_rows = []

for dataset_name, records in normalized_data.items():
    for record in records:
        text_value = str(record.get("original_text_preserved", "")).strip()

        if len(text_value) < 50:
            short_chunk_rows.append({
                "dataset": dataset_name,
                "chunk_id": (
                    record.get("chunk_id")
                    or record.get("id")
                    or record.get("source_record_id")
                ),
                "document_id": record.get("document_id"),
                "article_number": record.get("article_number"),
                "length": len(text_value),
                "text_preview": text_value[:200],
            })

short_chunks_df = pd.DataFrame(short_chunk_rows)

print("عدد المقاطع التي يقل طولها عن 50 حرفًا:", len(short_chunks_df))

if not short_chunks_df.empty:
    display(
        short_chunks_df.sort_values(
            ["dataset", "length"]
        ).reset_index(drop=True)
    )
else:
    print("لم يتم العثور على مقاطع أقصر من 50 حرفًا.")


عدد المقاطع التي يقل طولها عن 50 حرفًا: 4


,dataset,chunk_id,document_id,article_number,length,text_preview
0,ar_article,pdpl_ar_032__article_section__001,pdpl,32,6,ملغاة.
1,ar_article,transfer_regulation_ar_003__article_section__002,transfer_regulation,3,36,حماية البيانات الشخصية خارج المملكة.
2,ar_fixed,pdpl_ar_032__fixed_size__001,pdpl,32,6,ملغاة.
3,en_article,pdpl_032_chunk_01,pdpl,32,9,repealed.


### استنتاج المرحلة الخامسة: فحص المقاطع القصيرة

أظهر الفحص وجود أربعة مقاطع يقل طول كل منها عن 50 حرفًا:

- مقطعان يحتويان على النص `ملغاة.` للمادة 32 في مجموعتي التقسيم العربية.
- مقطع إنجليزي يحتوي على النص `repealed.` للمادة 32.
- مقطع عربي قصير بعنوان متعلق بحماية البيانات الشخصية خارج المملكة.

لم تُحذف هذه المقاطع؛ فقصر النص لا يعني بالضرورة انخفاض قيمته القانونية أو أهميته للاسترجاع. وستُحفظ معرّفاتها وبياناتها الوصفية لمراجعتها عند الحاجة.

**الخطوة التالية:** التحقق من اكتمال معرّفات المقاطع وعدم تكرارها داخل كل مجموعة.

In [20]:

# المرحلة 1.6: التحقق من معرّفات المقاطع

id_check_rows = []

for dataset_name, records in normalized_data.items():
    chunk_ids = [
        record.get("chunk_id")
        for record in records
    ]

    # بعض المجموعات تستخدم id بدل chunk_id
    chunk_ids = [
        chunk_id if chunk_id else record.get("id")
        for chunk_id, record in zip(chunk_ids, records)
    ]

    valid_ids = [
        str(chunk_id).strip()
        for chunk_id in chunk_ids
        if chunk_id is not None and str(chunk_id).strip()
    ]

    id_check_rows.append({
        "dataset": dataset_name,
        "total_chunks": len(records),
        "missing_ids": len(records) - len(valid_ids),
        "unique_ids": len(set(valid_ids)),
        "duplicate_id_records": len(valid_ids) - len(set(valid_ids)),
    })

id_check_df = pd.DataFrame(id_check_rows)
display(id_check_df)

print(
    "\nإجمالي المقاطع:",
    sum(row["total_chunks"] for row in id_check_rows)
)
print(
    "إجمالي المعرّفات المفقودة:",
    sum(row["missing_ids"] for row in id_check_rows)
)
print(
    "إجمالي سجلات المعرّفات المكررة داخل المجموعات:",
    sum(row["duplicate_id_records"] for row in id_check_rows)
)


,dataset,total_chunks,missing_ids,unique_ids,duplicate_id_records
0,ar_fixed,241,0,241,0
1,ar_article,196,0,196,0
2,en_fixed,181,0,181,0
3,en_article,252,0,252,0



إجمالي المقاطع: 870
إجمالي المعرّفات المفقودة: 0
إجمالي سجلات المعرّفات المكررة داخل المجموعات: 0


### استنتاج المرحلة السادسة: التحقق من المعرّفات

يهدف هذا الفحص إلى التأكد من وجود معرّف لكل مقطع، والتحقق من عدم تكرار المعرّفات داخل مجموعة البيانات نفسها.

وفق الملخص الإجمالي، تحتوي مجموعات البيانات الأربع على 870 مقطعًا، مع عدم رصد معرّفات مفقودة أو سجلات مكررة داخل المجموعات.

تُستخدم هذه المعرّفات لتمييز المقاطع وربطها بنتائج الاسترجاع لاحقًا. ولا يعني عدم وجود تكرار داخل كل مجموعة بالضرورة أن المعرّفات فريدة بين المجموعات الأربع جميعها.

**الخطوة التالية:** حفظ البيانات المجهزة وجداول الفحص في مجلد المخرجات، ثم التحقق من سلامة الملفات المحفوظة.

## 7. حفظ البيانات المجهزة ونتائج الفحص

### الهدف
حفظ مجموعات البيانات بعد تجهيز النصوص والبيانات الوصفية، إلى جانب جداول مقارنة استراتيجيات التقسيم وفحص المقاطع القصيرة ومعرّفاتها.

ستُحفظ الملفات في مجلد مستقل داخل `/kaggle/working/stage1_outputs`، مع التحقق من إمكانية قراءة ملفات JSON المحفوظة ومطابقة عدد سجلاتها للمجموعات الأصلية.

In [21]:
import json
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/stage1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# حفظ مجموعات البيانات الأربع والتحقق منها
for dataset_name, records in normalized_data.items():
    records_to_save = (
        article_data[dataset_name]
        if dataset_name in article_data
        else records
    )

    output_path = OUTPUT_DIR / f"{dataset_name}.json"

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(records_to_save, f, ensure_ascii=False, indent=2)

    with output_path.open("r", encoding="utf-8") as f:
        saved_records = json.load(f)

    assert len(saved_records) == len(records), (
        f"عدد السجلات غير متطابق في {dataset_name}"
    )

    print(f"{dataset_name}: تم الحفظ والتحقق — {len(saved_records)} مقطع")

# حفظ جداول نتائج الفحص
comparison_df.to_csv(
    OUTPUT_DIR / "chunking_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

id_check_df.to_csv(
    OUTPUT_DIR / "id_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

short_chunks_df.to_csv(
    OUTPUT_DIR / "short_chunks_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nعدد ملفات المخرجات:", len(list(OUTPUT_DIR.iterdir())))
print("مسار الحفظ:", OUTPUT_DIR)
print("اكتملت عملية الحفظ والتحقق الأولي.")

ar_fixed: تم الحفظ والتحقق — 241 مقطع
ar_article: تم الحفظ والتحقق — 196 مقطع
en_fixed: تم الحفظ والتحقق — 181 مقطع
en_article: تم الحفظ والتحقق — 252 مقطع

عدد ملفات المخرجات: 7
مسار الحفظ: /kaggle/working/stage1_outputs
اكتملت عملية الحفظ والتحقق الأولي.


# الاستنتاج النهائي للمرحلة الأولى: تجهيز البيانات ومقارنة التقسيم

## ملخص ما تم إنجازه

تم تحميل مجموعات البيانات العربية والإنجليزية الخاصة بمشروع **PDPL Compliance Copilot**، وفحص بنيتها، وتجهيز النصوص للبحث مع الحفاظ على النصوص الأصلية.

شملت المرحلة الأعمال التالية:

1. تحميل أربع مجموعات من المقاطع القانونية ومجموعة `Gold QA` المخصصة للتقييم اللاحق.
2. تطبيع النصوص العربية لأغراض البحث، مع الاحتفاظ بالنص الأصلي في حقل مستقل.
3. تجهيز البيانات الوصفية لمجموعتي `Article-level` العربية والإنجليزية.
4. مقارنة استراتيجيات التقسيم من حيث عدد المقاطع وأطوال النصوص وتوفر أرقام المواد والمعرّفات.
5. فحص المقاطع القصيرة جدًا والاحتفاظ بها دون حذف تلقائي.
6. التحقق من معرّفات المقاطع، دون رصد معرّفات مفقودة أو مكررة داخل المجموعات الأربع.
7. حفظ مجموعات البيانات ونتائج الفحص والتحقق من إمكانية قراءة ملفات JSON المحفوظة ومطابقة أعداد سجلاتها.

## النتائج الرئيسية

- إجمالي المقاطع: **870 مقطعًا** موزعة على أربع مجموعات.
- إجمالي أسئلة `Gold QA` المحمّلة: **90 سجلًا**، محفوظة للاستخدام في مرحلة التقييم اللاحقة.
- المقاطع القصيرة جدًا: **4 مقاطع**، تم الاحتفاظ بها للمراجعة.
- المعرّفات: جميع المقاطع لها معرّفات، ولم تُرصد معرّفات مكررة داخل أي مجموعة.
- المخرجات المحفوظة: **7 ملفات** داخل `/kaggle/working/stage1_outputs`.

## حدود المرحلة

تقتصر هذه المرحلة على تجهيز البيانات وفحص بنيتها ومقارنة خصائص استراتيجيات التقسيم. ولا تثبت النتائج الوصفية وحدها أي استراتيجية تحقق أداء استرجاع أفضل، كما لم تشمل هذه المرحلة بناء نظام الاسترجاع أو توليد الإجابات أو التقييم النهائي باستخدام `Gold QA`.

## المخرجات والخطوة التالية

تم تجهيز البيانات الأولية وحفظها لتكون أساسًا للمراحل اللاحقة من المشروع.

**المرحلة التالية:** بناء خط استرجاع للمقاطع القانونية، ثم تقييم النتائج باستخدام أسئلة التقييم المرجعية المؤهلة، مع مراعاة اللغة والمراجع القانونية وصحة الاستشهادات.

In [22]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/stage1_outputs")

files = sorted(OUTPUT_DIR.iterdir())

print("المجلد موجود:", OUTPUT_DIR.is_dir())
print("عدد الملفات:", len(files))

for file in files:
    print(f"{file.name} — {file.stat().st_size:,} bytes")

assert len(files) == 7, "تحققي من عدد الملفات"
print("\nتم التحقق من وجود الملفات السبعة.")

المجلد موجود: True
عدد الملفات: 7
ar_article.json — 1,597,046 bytes
ar_fixed.json — 1,659,265 bytes
chunking_comparison.csv — 353 bytes
en_article.json — 1,009,549 bytes
en_fixed.json — 961,220 bytes
id_validation.csv — 156 bytes
short_chunks_review.csv — 396 bytes

تم التحقق من وجود الملفات السبعة.


In [23]:
import shutil
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/stage1_outputs.zip"

shutil.make_archive(
    "/kaggle/working/stage1_outputs",
    "zip",
    root_dir="/kaggle/working",
    base_dir="stage1_outputs"
)

display(FileLink(ZIP_PATH))

/kaggle/working/stage1_outputs.zip

In [24]:
from pathlib import Path

output_dir = Path("/kaggle/working/stage1_outputs")

print("المجلد موجود:", output_dir.exists())

if output_dir.exists():
    for file in sorted(output_dir.iterdir()):
        print(file.name, f"({file.stat().st_size:,} bytes)")

المجلد موجود: True
ar_article.json (1,597,046 bytes)
ar_fixed.json (1,659,265 bytes)
chunking_comparison.csv (353 bytes)
en_article.json (1,009,549 bytes)
en_fixed.json (961,220 bytes)
id_validation.csv (156 bytes)
short_chunks_review.csv (396 bytes)


In [ ]:
from pathlib import Path
import shutil

output_dir = Path("/kaggle/working/stage1_outputs")
zip_path = shutil.make_archive(
    "/kaggle/working/stage1_outputs",
    "zip",
    root_dir=output_dir
)

print("تم إنشاء الملف:", zip_path)
print("حجم الملف:", Path(zip_path).stat().st_size, "bytes")